# Create a Conversational Analytics data agent

Run these steps on Google Colab to create a Conversational Analytics data agent for the Looker Embed Reference.

You must run **ALL** code sections **in order**.

*This notebook is only meant to be run with the [Looker Embed Reference](https://github.com/looker-open-source/LookerEmbedReference) or codelab.*

## Prerequisites

* Have a Google Cloud project
* These [APIs](https://docs.cloud.google.com/gemini/docs/conversational-analytics-api/enable-the-api#required-apis) are enabled in your Cloud project:
    * `geminidataanalytics.googleapis.com`
    * `bigquery.googleapis.com`
    * `cloudaicompanion.googleapis.com`
* You may also need these [APIs](https://docs.cloud.google.com/gemini/docs/conversational-analytics-api/enable-the-api#apis-for-colab-enterprise) enabled.
* You are running this Python notebook on Google Colab
* You are logged into a Cloud user account on Google Colab that has these permissions:
    * `roles/cloudaicompanion.user`
    * `roles/looker.instanceUser`
    * `roles/bigquery.user`

## Steps

1. Set your Cloud project ID and Looker instance URI below:





In [ ]:
PROJECT_ID = "YOUR_PROJECT_ID"
# Format: "https://my.looker.app/" with a trailing slash
LOOKER_INSTANCE_URI_WITH_TRAILING_SLASH = "YOUR_LOOKER_INSTANCE_URL"

2. Setup your environment variables

**DO NOT CHANGE THE FOLLOWING VARIABLES IF YOU ARE FOLLOWING THE README OR CODELAB.**

*Only update the following environment variables if you want to create a data agent with a different ID, or point your data agent to a different Looker explore*

In [3]:
LOOKER_MODEL = "data_block_acs_bigquery"
LOOKER_EXPLORE_1 = "acs_census_data"
LOOKER_EXPLORE_2 = "congressional_district"
AGENT_ID = "looker_embed_reference_data_agent"

3. Install the Conversational Analytics Python *SDK*

In [3]:
%pip install google-cloud-geminidataanalytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 290.2/290.2 kB 9.2 MB/s eta 0:00:00


4. Auth into Google Cloud and initialize the Conversational Analytics data agent client.

In [4]:
import time
from google.colab import auth
from google.cloud import geminidataanalytics_v1beta as geminidataanalytics

auth.authenticate_user(project_id=PROJECT_ID)
data_agent_client = geminidataanalytics.DataAgentServiceClient()


5. Create the data agent pointing to your Looker instance's explore

In [5]:

looker_explore_reference_1 = geminidataanalytics.LookerExploreReference(
  looker_instance_uri=LOOKER_INSTANCE_URI_WITH_TRAILING_SLASH,
  lookml_model=LOOKER_MODEL,
  explore=LOOKER_EXPLORE_1,
)

looker_explore_reference_2 = geminidataanalytics.LookerExploreReference(
  looker_instance_uri=LOOKER_INSTANCE_URI_WITH_TRAILING_SLASH,
  lookml_model=LOOKER_MODEL,
  explore=LOOKER_EXPLORE_2,
)

datasource_references = geminidataanalytics.DatasourceReferences(
  looker=geminidataanalytics.LookerExploreReferences(
    explore_references=[looker_explore_reference_1, looker_explore_reference_2],
  ),
)

published_context = geminidataanalytics.Context(
  system_instruction="",
  datasource_references=datasource_references,
  options=geminidataanalytics.ConversationOptions(
    chart=geminidataanalytics.ChartOptions(
      image=geminidataanalytics.ChartOptions.ImageOptions(
        svg={}
      )
    )
  ),
)

data_agent = geminidataanalytics.DataAgent(
  data_analytics_agent=geminidataanalytics.DataAnalyticsAgent(
    published_context=published_context
  ),
)

request = geminidataanalytics.CreateDataAgentRequest(
  parent=f"projects/{PROJECT_ID}/locations/global",
  data_agent=data_agent,
  data_agent_id=AGENT_ID,
)

try:
  operation = data_agent_client.create_data_agent(request=request)

  while not operation.done():
    print("Still in progress...")
    time.sleep(2)

  if operation.done():
    if operation.exception():
      print("Data agent creation operation failed:", operation.exception())
    else:
      print("Data agent created:", operation.result())
except Exception as e:
  print(f"Error creating Data Agent: {e}")

Error creating Data Agent: 409 Resource 'projects/next25-workshop/locations/global/dataAgents/looker_embed_reference_data_agent' already exists [resource_name: "projects/next25-workshop/locations/global/dataAgents/looker_embed_reference_data_agent"
]
